# Results 02 · Incremental information

What does the zigzag topology add beyond the activity level? For each problem, the score of **topology + control** minus the score of the **control alone** (same folds, same threshold). > 0 means the topology carries information the level control does not. Topology = all Turnover dimensions (H0+H1+H2), H1 only, or H2 only.

**Control.** Active fraction (AF) at the same zigzag threshold. For per-frame thresholds (pfXX) the active fraction is constant by construction, so the control is the per-frame moments of the grid (mean, std, skewness across cells).

**Classifier.** Logistic regression with the regularization strength chosen by an inner cross-validation on the training data only (`logreg_cv`), for every feature set, so that adding features does not simply overfit.

**Score.** Chance-normalized balanced accuracy, (bacc − 1/K) / (1 − 1/K), with K the number of classes of each
problem: 0 = chance, 1 = perfect. This makes problems with different numbers of classes comparable.

**One point = one classification problem**: a mouse × stimulus label within mouse, a held-out mouse (video
label, segment ID) or a directed mouse pair (video ID) across mice. 3D-CNN repeats are averaged per problem.
Only complete runs are used (set `INCLUDE_PARTIAL = True` to include runs in progress), and for each
configuration the latest run.

**Abbreviations.** AF = active fraction (fraction of grid points above the zigzag threshold, per frame).
moments = mean/CV/skewness (min-max grid) or mean/std/skewness (trial-mean grid) across grid cells, per frame;
"dens." = cells divided by neuron density; "avg" = averaged over the trial window (overall level, no time course).
pXX = per-trial percentile threshold P_ACTIVE; abs0 = fixed threshold 0 (trial-mean grid).

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import results_viz as rv

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)

# Settings
INCLUDE_PARTIAL = False   # True: also show runs still in progress (marked in the inventory)
SAVE_FIGURES = True       # PNG + PDF in notebooks/figures/results/
MODEL = rv.TUNED_MODEL    # 'logreg_cv' (tuned C); 'logreg' shows the earlier fixed-C runs

df = rv.load_all(include_partial=INCLUDE_PARTIAL)
pp = rv.per_problem(df) if not df.empty else df
print(f"{len(df)} result rows, {len(pp)} problems x feature sets; grids: {sorted(df['grid'].unique()) if len(df) else []}")
print("thresholds found:", rv.thresholds(df) if len(df) else [])

pp = rv.select_models(pp, [MODEL])
deltas = rv.incremental_deltas(pp)
print(deltas.groupby(['grid', 'comparison']).size() if len(deltas) else 'no paired results yet')

## Topology + control vs control

In [ ]:
fig = rv.plot_incremental(deltas, 'min-max grid')
if fig is not None and SAVE_FIGURES:
    rv.save(fig, '02_incremental_minmax')

In [ ]:
fig = rv.plot_incremental(deltas, 'trial-mean grid')
if fig is not None and SAVE_FIGURES:
    rv.save(fig, '02_incremental_trialmean')

### Statistics

Wilcoxon signed-rank test of the per-problem differences, per grid × threshold × scheme × topology subset, Holm-corrected over all tests in the table. With fewer than 6 problems (video label: 5 mice) a two-sided test cannot reach p < 0.05; those rows have no p-value.

In [ ]:
if len(deltas):
    stats = rv.wilcoxon_table(deltas[deltas['comparison'].str.startswith('topology + control')],
                              ['grid', 'thr', 'target', 'scope', 'subset'])
    display(stats.round(4))

## Topology alone vs control

Same comparison without combining: is the topology on its own better or worse than the level control?

In [ ]:
fig = rv.plot_incremental(deltas, 'min-max grid', comparison='topology alone  vs  control')
if fig is not None and SAVE_FIGURES:
    rv.save(fig, '02_alone_minmax')

In [ ]:
fig = rv.plot_incremental(deltas, 'trial-mean grid', comparison='topology alone  vs  control')
if fig is not None and SAVE_FIGURES:
    rv.save(fig, '02_alone_trialmean')

## Components per threshold

The raw scores behind the differences: control alone, each topology subset alone, and combined.

In [ ]:
for grid in ['min-max grid', 'trial-mean grid']:
    for thr in rv.thresholds(pp[pp['grid'] == grid]) if len(pp) else []:
        fig = rv.plot_components(pp, grid, thr)
        if fig is not None and SAVE_FIGURES:
            rv.save(fig, f"02_components_{grid.split()[0]}_{thr}")